# Machine learning volatility models

- Notes from relevant papers
- Focus on what info may be needed for project
- Summary/thoughts at bottom

# [ML Approach to volatility forcasting](https://arxiv.org/pdf/2601.13014)

## Overview

- Complicated structure of data makes HAR inadequate sometimes 
- Volatility predicted by past volatility but there are other covariates 
- Paper looks at out-of-sample performance of linear regularized models (ridge, lasso, elastic net), tree-based algorithms (bagging, random forest, gradient boosting), and neural networks in high-dimensional data setting 
- Regularization helps deal with high dimensionality and gets less noisy parameter estimates 
- Tree-based regression allows for estimation without imposing distributional/functional form on model 
- Neural networks can regularize and capture nonlinearities 

## Setup 

Let $X = (X_t)_{t\geq 0}$ be log-price, supported on probability space $(\Omega, (F_t)_{t\geq 0}, F,P)$. Assume 
$$ X_t = X_0 + \int_0^t\mu_s ds+\int_0^t\sigma_s dW_s+\sum_{s=1}^{N_t}J_s $$ 
where $X_0$ is $F_0$-measurable, $\mu = (\mu_t)$ is the drift, $\sigma = (\sigma_t)$ the stochastic volatility process, $W = (W_t)$ std Brownian motion, $N = (N_t)$ a counting process representing number of jumps in $X$, and $J=(J_s)_{s=1}^{N_t}$ a sequence of nonzero random variables of jump sizes with jup times $\tau = (\tau_s)$ 

**Goal**: forecast *daily quadratic variation* $QV_t = \int_{t-1}^t\sigma_s^2ds+\sum_{\tau_s=t-1}^{t}J_s^2$ for $t = 1,..,T$ where $T$ is total number of days in rample. 

$QV$ is estimated by *realized variance* 
$$RV_t=\sum_{j=1}^n|\Delta^n_{t-1,j}X|^2 \text{ where } \Delta_{t-1,j}^nX = X_{t-1+j/n}-X_{t-1+(j-1)/n}$$ 
and $n$ is number of intraday log returns. $RV_t \to QV_t$ as $n\to\infty$. Can decompose RV into positive and negative semivariance part

### Regularization

To avoid overfitting, shrink regression coefficients. 

The *penalized loss function* is 
$$\tilde{L}(\beta_0,\beta;\theta) = L(\beta_0,\beta)+\phi(\beta;\theta)$$ 
where $\phi(\beta;\theta)$ is a penalty term, $\theta$ a vector of hyperparameters determined in validation set, and $L$ is the loss function. 

For *ridge regression*, 
$$\phi(\beta;\lambda) = \lambda\sum_{i=1}^J\beta_i^2$$ 
where $\lambda \geq 0$ controls amount of shrinkage. 

For a large feature space use Lasso, with 
$$\phi(\beta;\lambda) = \lambda\sum_{i=1}^J|\beta_i|$$ 
Lasso often forces coefficient estimates to zero and generates sparsity and subset selection, so there is no closed-form solution to the problem. 

For elastic net, 
$$\phi(\beta;\lambda,\alpha) = \lambda\left(\alpha\sum_{i=1}^J\beta_i^2+(1-\alpha)\sum_{i=1}^J|\beta_i|\right)$$ 
with $\alpha\in[0,1]$

### Tree-based regression 

With linear models have to impose true association among predictors and response variables. A regression tree is fully nonparametric so can implicitly account for interactions and can be nonlinear 

Based on a decision tree: Partition domain (feature space) of explanatory variables $Z_t$ with its domain in $R^J$ into successively smaller rectangular subspaces as one moves through the tree until a stopping criterion is reached. A constant prediction is assigned to all observations in a given terminal node. 

Suppose the tree has $M$ leaves, corresponding to a sequence of disjoint rectangles $R_m$ which partition the domain of $Z_t$. Tree-based regression then predicts 
$$f(Z_t) = \sum_{m=1}^M C_m\delta_{Z_t\in R_m}$$ 
where $C_m$ is a constant, ex. $C_m = avg(RV_{t+1} : Z_t \in R_m)$ 

**Random forest**: restrict to random subset of input features and find best split point. Grow $B$ trees $\{f(Z_t^b,\theta^b)\}_{b=1}^B$ and predict with $f(Z_t) = \frac{1}{B}\sum_{b=1}^Bf^b(Z_t,\theta^b)$ 

**Graddient boost**: sequential model based on weak learners. Each tree is grown on info from previous one. Initialize $f^0(Z_t) = \arg\min\sum_{t}(RV_t-\beta_0)^2$ as a constant. Use negative gradiant of loss function wrt prediction. Fit a shallow tree to the residuals to get a set of terminal nodes $R_{j,b}$ for $j$ the leaf and $b$ the tree. Chose a gradient descent size $\rho^{jb} = \arg\min_{\beta_0}\sum_{Z_t\in R_{jb}}(RV_t-\beta_0-f^{b-1}(Z_t))^2$. Update recursively: 
$$f^b(Z_t) = f^{b-1}(Z_t)+\nu\sum_{j=1}^{J_b}\rho^{jb}1_{Z_t\in R_{jb}}$$ for $\nu$ learning rate. 

GB deals with underfitting problem, so places more weight on misclassification and is susceptible to outliers. RF deals with overfitting.

### Neural Networks 

Very flexible and nonlinear, but has a lot of hyperparameters. Need regularization to avoid overfitting. Helpful to have additional hidden layers to increase efficiency 

### Forecast comparison 

Use an out-of-sample evaluation measure $$L(f(Z_t)) = \sum_t(RV_{t+1}-f(Z_t))^2$$ where $f(Z_t)$ is the forecast. 

If volatility is negative, use the min in-sample realized variance instead. 

Use pairwise Diebold and Mariano test to measure statistical significance and use model confidence set to find "best" models.

## Data

Used continuous stock price record from 2001 to 2017 from 29 companies. Sample includes financial crisis, European sovereign debt crisis, etc. Data from NYSE TAQ database. Preprocessed and filteres for outliers. A five-minute log-return series used and computed time series for daily RV for each asset. Use 70/10/20 split for training/validation/test. Most hyperparameters set equal to default values. 


# [Stochastic Volatility Modelling with LSTM Networks](https://arxiv.org/pdf/2512.12250)

Compare stochastic volatility (SV), long short-term memory (LSTM) NN, and hybrid SV-LSTM 

## Data

Used SP 500 index with data from yfinance from 1998 to 2024. Forecasting target is rolling historical volatility computed over a window of $N=21$ days, estimated by 
$$\sigma_t = \sqrt{\frac{1}{N-1}\sum_{i=t-N+1}^t(r_i-r_t)^2}$$ 
for $\sigma_t$ rolling volatility at time $t$, $N$ window size, $r_i$ log return at time $i$, $r_t = \frac{1}{N}\sum_{i=t-N+1}^tr_i$ rolling mean of log returns. Use this to evaluate predictive accuracy of models. 

### Preprocessing

For LSTM rescale data for numerical stability: 
$$X_t^s = \frac{X_t-\min(X)}{\max(X)-\min(X)}(1-10^{-11})+10^{-11}$$ 
for $X$ the set of log returns within each window and $10^{-11}$ used to avoid division by zero/ensure rescaled values are between $10^{-11}$ and $1$. 

Two-step scaling: scaled training/validation sets together for consistent range, then training set scaled seperately. After predictions generated, scalers applied to test data stored and used to inverse-transform predicted values back to original scale. 

### Evaluation metrics 

**Mean squared error**: quantifies avg squared difference bw actual vs predicted rolling volatility values, 
$$MSE = \frac{1}{n}\sum_{i=1}^n(y_i-\hat{y}_i)^2$$ 
for $y_i$ actual volatility and $\hat{y}_i$ predicted volatility and $n$ number of observations in rolling window. 

**Mean absolute error**: avg magnitude of errors, 
$$MAE = \frac{1}{n}\sum_{i=1}^n|y_i-\hat{y}_i|$$ 

**Mean absolute percentage error**: error relative to volatility as percentage, 
$$MAPE = \frac{1}{n}\sum_{i=1}^n|\frac{y_i-\hat{y_i}}{y_i}|\times 100$$ 

## LSTM 

*Input*: log returns, 21 day historical volatility between 2000-2025 
*Output*: rolling volatility at $t+1$ 

Used multi-year rolling window for training: each window used 11 years of training data, 3 years of validation data, 1 year of test data. After each test period, window shifted forward by one year LSTM good for time series forecasting, capturing long-term dependencies, adapting to volatile shocks. 

Has memory cells and forget, input, and outpute gates. 

*Forget gate*: $f_t = \sigma(U_fx_t+V_fh_{t-1}+b_f)$ where $h_{t-1}$ is previous hidden state, $x_t$ the current input, $b_f$ the bias vector. Uses a sigmoid activation function to determine how much of previous cell state $C_{t-1}$ to retain. Output values between zero and one, and $C_t' = f_tC_{t-1}$. 

*Input gate*: $i_t = \sigma(U_ix_t+V_ih_{t-1}+b_i)$ decides how much new info to add to cell: $C_t^+=\tanh(U_cx_t+V_xh_{t-1}+b_c)$. 

Cell updated as $$C_t = C_t'+i_tC_t^+$$ 

*Output gate*: $o_t = \sigma(U_ox_t+V_oh_{t-1}+b_o)$ shapes hiddent state with $h_t=o_t\tanh(C_t)$ with $o_t \in (0,1)$. 

During training, loss function is minimized by adjdusting parameters based on error bw predictions/values using gradient descent with update rule 
$$w_t = w_{t-1}-\eta\frac{\partial L}{\partial w}$$ 
for $\eta$ learning rate, $\frac{\partial L}{\partial w}$ gradient of loss wrt parameter. Increase in validation error despite decreasing training loss implies over-specialization, hence training should be stopped to avoid overfitting. 

Hyperparameter tuning done with 25 random seach combinations, tested across three trials.

## Statistical tests 

Used Wilcoxon signed-rank test and diebold-mariano test to assess forecasting error differences between tested models. 

*Wilcoxon test*: Checks if models' error distributions similar without assuming a specific shape. Calculate differences between paired errors, rank them and check to see if difference is statistically significant 

*Diebold-Mariano test*: Checks which model forecasts more accurately over time. Compute differences between loss function results 


# Overall Thoughts

- If dealing with outliers/trying to compare volatility models to see how robust they are should use RF instead of GB
- Elastic net combines ridge/lasso so maybe should stick to that one
- Regressive techniques may be "too simple" ie results not too different than other models used
- Probably best to implement RF and if interested LSTM
- RF
    - scikit has RF classifier: [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
    - Can build it by hand: [tutorial](https://towardsdatascience.com/building-a-random-forest-by-hand-in-python-187ac0620875/)
    - RF for time series: [tutorial](https://machinelearningmastery.com/random-forest-for-time-series-forecasting/)
- LSTM
    - Pytorch: [documentation](https://docs.pytorch.org/docs/2.12/generated/torch.nn.LSTM.html)
    - Time series: [tutorial](https://machinelearningmastery.com/time-series-prediction-lstm-recurrent-neural-networks-python-keras/)
    - Using Keras: [tutorial](https://medium.datadriveninvestor.com/forecasting-real-time-market-volatility-using-python-282e78b61022) 
- Compare one day vs one week vs one month ahead predictions
- Use wide range of data, should include 2008, 2019-2023, etc. Perhaps 1998 - 2024